# Un lazo de control al que se le pueden mover las perillas

Un Arduino corre un lazo de control de 1 kHz sobre un motor con un sensor
magnético de ángulo. Este notebook fija las ganancias, corre un experimento y se
trae de vuelta la serie temporal.

Cada celda arranca con `sync_board()`, que recompila el sketch si se lo editó,
lo vuelve a grabar si cambió el binario, y reabre el enlace, lo que resetea la
placa. Así cada celda arranca desde los valores por omisión del sketch, y se
pueden correr en cualquier orden sin preguntarse qué dejó la celda de arriba.

**El motor se va a mover.** Revisar el banco antes de correr cualquier cosa de
las que siguen.

El controlador propiamente dicho está en `../ControlDemo/ControlDemo.ino`. Ése es
el archivo que hay que editar para cambiar la *ley* de control; todo lo de acá
sólo cambia sus parámetros.

---

**Antes de la primera corrida.** Hace falta tener el [Arduino CLI](https://arduino.github.io/arduino-cli/)
en el PATH y el core de AVR instalado:

```
arduino-cli core install arduino:avr
pip install -r ../python/requirements.txt
```

Windows, macOS y Linux funcionan por igual, y la placa se encuentra sola: no
debería hacer falta nombrar ningún puerto. Si el Arduino CLI se instaló con esta
terminal ya abierta, cerrarla y volver a abrirla: el PATH se lee una sola vez al
arrancar, y en Windows ésa es la razón habitual de que `sync_board()` no lo
encuentre.

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt

from bench import *          # sync_board, y las unidades de este equipo

plt.rcParams['figure.figsize'] = (9, 3.5)
plt.rcParams['axes.grid'] = True

## 1. ¿Funciona el hardware?

Conviene correr esto antes que nada, y de nuevo después de tocar el cableado.
Verifica por separado cada parte del equipo —la temporización del lazo, el imán,
el bus I2C, la medición de corriente y por último el motor—, de modo que una
falla apunte a una sola cosa y no a "el experimento no anduvo".

Al final hace girar el motor un instante. Pasar `motor=False` para saltear eso.

In [ ]:
dev = sync_board()
dev.bringup()

## 2. El sensor

`y_uw` es el ángulo del eje, desenrollado: sigue contando a través de la vuelta
de 4096 cuentas en lugar de saltar de vuelta a cero, así que un eje que gira sin
parar da una recta que sube sin parar. Girar el imán con la mano mientras esto
corre.

`y_uwf` es la misma señal pasada por un filtro de dos polos. `dev.smooth('y', tau)`
fija su constante de tiempo; `tau = 0` lo apaga, que es lo que viene por omisión:
50 ms de retardo es mucha fase para regalar en un lazo de 1 kHz, y siempre se
puede filtrar de este lado después.

In [ ]:
dev = sync_board()
dev.zero()
dev.smooth('y', 0.05)           # 50 ms

df = dev.capture(3.0)

plt.plot(df['t'], df['y_uw'],  lw=0.8, label='y_uw   crudo')
plt.plot(df['t'], df['y_uwf'],          label='y_uwf  filtrado')
plt.xlabel('t [s]'); plt.ylabel('ángulo [grados]'); plt.legend()
plt.title('girar el imán')
plt.show()

print(f'se movió {df["y_uw"].max() - df["y_uw"].min():.1f} grados, '
      f'ruido {df["y_uw"].diff().std():.3f} grados entre muestras')

## 3. Lazo abierto: ¿qué hace la planta?

`mode = MODE_OPEN` desconecta el controlador y pone `uff` directamente sobre el
motor. Aplicarle un escalón y mirar la respuesta. Esa respuesta —cuán rápido
acelera, cuánta corriente consume— es contra lo que hay que diseñar un
controlador, así que ésta es la primera medición que hay que tomar.

`dev.step()` mantiene `pre` segundos, cambia el parámetro, y después mantiene
`post` segundos más. La placa informa el tick exacto en el que cayó el cambio,
así que `t = 0` es el escalón mismo con precisión de una muestra; la fluctuación
de temporización de este lado nunca entra en los datos.

In [ ]:
dev = sync_board()
dev.mode = MODE_OPEN

df = dev.step('uff', 200, pre=0.3, post=0.7, back=0)
dev.rest()

velocidad = np.diff(df['y_uw']) / (df.attrs['dt_us'] * 1e-6) / 360

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6))
a.plot(df['t'][1:], velocidad); a.set_ylabel('velocidad [vueltas/s]')
b.plot(df['t'], df['u']);       b.set_ylabel('u [pwm]')
c.plot(df['t'], df['i']);       c.set_ylabel('i [mA]'); c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('escalón en lazo abierto: u = 0 -> 200')
plt.show()

## 4. Cerrar el lazo sobre la posición

`mode = MODE_PID` pone en marcha el controlador, y `target = POSITION` hace que
trabaje sobre el ángulo. El error es `ref - y_uw`, los dos en cuentas;
`dev.deg()` permite escribir una referencia en grados.

`dev.gains(kp, ki, kd)` toma las ganancias en tiempo continuo —`ki` por segundo,
`kd` en segundos— y las convierte. Los `dev.kp`, `dev.ki` y `dev.kd` propios de
la placa son por *muestra*, que es por lo que su aritmética realmente multiplica.

Conviene empezar sólo con kp. Agregar ki recién cuando se vea un error de régimen
permanente que valga la pena eliminar, y kd sólo si se ve una oscilación que
valga la pena amortiguar.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.gains(kp=0.002, ki=0.0, kd=0.0)

dev.zero()
dev.ref  = 0
dev.mode = MODE_PID

df = dev.step('ref', dev.deg(90), pre=0.2, post=0.8, back=0)
dev.rest()

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], dev.as_deg(df['ref']), 'k--', lw=0.8, label='ref')
a.plot(df['t'], df['y_uw'], label='y_uw')
a.set_ylabel('ángulo [grados]'); a.legend()
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('escalón en lazo cerrado a 90 grados')
plt.show()

final = df['y_uw'].iloc[-1]
print(f'terminó en {final:.1f} grados, {90 - final:+.1f} grados de error '
      f'de régimen permanente')

## 5. Barrer una ganancia

La razón de manejar todo esto desde un notebook: cambiar un número, volver a
medir, superponer. Cada pasada es un viaje de ida y vuelta a la placa.

Observar qué compra y qué cuesta subir kp: velocidad contra sobrepico, y
finalmente contra oscilación.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.zero()
dev.ref  = 0
dev.mode = MODE_PID

corridas = {}
for kp in (0.0005, 0.001, 0.002, 0.004):
    dev.gains(kp=kp)
    corridas[kp] = dev.step('ref', dev.deg(90), pre=0.1, post=0.6, back=0)

dev.rest()

for kp, d in corridas.items():
    plt.plot(d['t'], d['y_uw'], label=f'kp = {kp}')
plt.axhline(90, color='k', lw=0.8, ls='--')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uw [grados]'); plt.legend()
plt.title('barrido de kp')
plt.show()

for kp, d in corridas.items():
    despues = d[d['t'] > 0]['y_uw']
    print(f'kp={kp:<7} pico {despues.max():6.1f} grados, '
          f'final {despues.iloc[-1]:6.1f} grados')

## 6. Seguir una rampa

`mode = MODE_RAMP` le suma `refrate` a `ref` en cada período de control, así que
la referencia barre a velocidad constante y el lazo tiene que *seguirla* en lugar
de establecerse.

Un controlador proporcional no puede seguir una rampa sin quedarse atrás: el
error es lo que genera el comando, así que un comando constante necesita un error
constante. Agregar ki es lo que cierra esa brecha. Conviene probar primero con
`ki=0` y mirar el atraso.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.gains(kp=0.002, ki=0.05)

dev.zero()
dev.ref     = 0
dev.refrate = dev.rev_per_s(1.0)
dev.mode    = MODE_RAMP

df = dev.step('refrate', dev.rev_per_s(2.0), pre=2.0, post=2.0)
dev.rest()

por_s = 1 / (df.attrs['dt_us'] * 1e-6) / 360

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'][1:], np.diff(dev.as_deg(df['ref'])) * por_s, 'k--', lw=0.8, label='ref')
a.plot(df['t'][1:], np.diff(df['y_uw']) * por_s, label='y_uw')
a.set_ylabel('velocidad [vueltas/s]'); a.legend()
b.plot(df['t'], dev.as_deg(df['e'])); b.set_ylabel('e [grados]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('rampa: 1 vuelta/s, después 2')
plt.show()

## 7. El mismo controlador, otra señal

`target = CURRENT` cambia la realimentación: el mismo PID ahora trabaja sobre
`ref - i`, la corriente que circula por el motor. Nada más cambia —en el sketch
es una rama en `target_error()`—, así que éste es el mismo controlador contra una
planta mucho más rápida, y necesita otras ganancias exactamente por eso.

`dev.ma()` escribe la referencia en miliamperes. La corriente es ruidosa, así que
`alpha_i` filtra la medición y `alpha_e` filtra lo que ve el término derivativo;
`dev.smooth()` fija cualquiera de los dos por constante de tiempo.

In [ ]:
dev = sync_board()

dev.target = CURRENT
dev.smooth('i', 0.005)
dev.smooth('e', 0.010)
dev.gains(kp=0.05, ki=2.0)

dev.ref  = dev.ma(150)
dev.mode = MODE_PID

df = dev.step('ref', dev.ma(300), pre=0.5, post=0.5, back=dev.ma(150))
dev.rest()

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], dev.as_ma(df['ref']), 'k--', lw=0.8, label='ref')
a.plot(df['t'], df['i'], label='i')
a.set_ylabel('corriente [mA]'); a.legend()
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('escalón de corriente, 150 -> 300 mA')
plt.show()

## 8. Qué hace la frecuencia de muestreo

`tickdiv` divide el muestreador de 5 kHz hasta la frecuencia del lazo:
`tickdiv = 5` da 1 kHz, `tickdiv = 50` da 100 Hz. Las ganancias del sketch son
*por muestra*, así que los mismos tres números significan algo distinto a cada
frecuencia, y eso es justamente el punto. Un lazo ajustado a 1 kHz y después
corrido a 100 Hz es un lazo con la décima parte de la acción integral y diez
veces la ganancia derivativa.

`dev.gains()` vuelve a convertir desde tiempo continuo, así que llamarlo de nuevo
después de cambiar `tickdiv` deja el lazo donde estaba. Comentarlo para ver qué
pasa si uno se olvida.

In [ ]:
dev = sync_board()

dev.target = POSITION

for tickdiv in (5, 25, 50):
    dev.tickdiv = tickdiv
    dev.gains(kp=0.002, ki=0.05)      # reconvertidas para el nuevo período
    dev.zero()
    dev.ref  = 0
    dev.mode = MODE_PID

    d = dev.step('ref', dev.deg(90), pre=0.1, post=0.6, back=0)
    plt.plot(d['t'], d['y_uw'], label=f'{1e6 / d.attrs["dt_us"]:.0f} Hz')

dev.rest()

plt.axhline(90, color='k', lw=0.8, ls='--')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uw [grados]'); plt.legend()
plt.title('las mismas ganancias a tres frecuencias de lazo')
plt.show()

## 9. Para terminar

Dejar el motor en reposo y liberar el puerto, o el próximo `sync_board()` lo va a
encontrar ocupado.

In [ ]:
dev = sync_board()
dev.rest()
dev.close()
print('cerrado')